# Preprocesamiento y comparativa RNN vs LSTM vs GRU
## Dataset: ElectricityLoadDiagrams20112014 (UCI)


Este cuaderno esta construido tomando como base directa el cuadernillo
`02_rnn_forecast.py` (el que en su momento explicaba como predecir el ultimo valor de
una serie temporal sintetica con una RNN). Vamos a reutilizar la misma estructura que
ahi se presenta: la clase `TimeSeriesDataset`, la forma de armar `dataset` y
`dataloader` como diccionarios con las particiones, la funcion `plot_series` para
visualizar resultados en una grilla, la funcion `fit` con barra de progreso, y la
funcion `predict`. La diferencia principal es que en vez de trabajar con una serie
sintetica generada con senos y ruido, vamos a trabajar con datos reales de consumo
electrico, y en vez de entrenar una unica RNN vamos a entrenar y comparar tres
arquitecturas recurrentes: RNN simple, LSTM y GRU.

Ademas, sobre esa base le sumamos todo lo que pide `guia_optimizacion_regularizacion.md`
(que a su vez resume `01_receta_entrenamiento`, `01_optimization` y
`01_regularization`): validar la red antes de entrenar en serio, iterar rapido sobre
un subconjunto antes de ir al dataset completo, usar checkpointing y early stopping,
aplicar weight decay y dropout, y dejar todo controlado por una funcion de
entrenamiento reutilizable.

El entrenamiento esta pensado para correrse **en local**, en una GPU tipo
**RTX 4070 de 8 GB de VRAM**.

### Estructura del cuaderno

**Parte 1 - Preprocesamiento**
1. Descarga y carga del dataset
2. Exploracion de datos (EDA)
3. Decision de diseno: que vamos a predecir, y como se cumplen `n > 20` y `m = o > 20.000`
4. Particion temporal train / eval / test y normalizacion
5. `TimeSeriesDataset`, `dataset` y `dataloader` (misma estructura que la base)
6. `plot_series`, adaptada a datos reales

**Parte 2 - Validando la red antes de entrenar en serio**
7. Modelo generico RNN / LSTM / GRU (basado en las clases `RNN` y `DeepRNN` del cuadernillo base)
8. Comprobacion de dimensiones, fit de una muestra, fit de un batch

**Parte 3 - Entrenamiento y comparacion (prediccion a un paso)**
9. Funcion `fit` ampliada con checkpointing, early stopping, scheduler y weight decay
10. Subconjunto representativo y busqueda de hiperparametros
11. Entrenamiento final de las tres arquitecturas sobre el dataset completo
12. Evaluacion en test y comparacion de resultados

**Parte 4 - Extensiones tomadas del cuadernillo base**
13. Prediccion a varios pasos (varias horas hacia adelante)
14. Intervalos de confianza usando dropout en la inferencia (MC-Dropout)

**Parte 5 - Conclusiones**

## 0. Imports y configuracion general

Las mismas librerias que usa el cuadernillo base (`torch`, `numpy`, `matplotlib`,
`tqdm` para la barra de progreso de entrenamiento, `sklearn.metrics` para el error),
mas lo necesario para descargar y leer el dataset real (`urllib`, `zipfile`, `pandas`)
y `StandardScaler` para normalizar, que en el cuadernillo original no hacia falta
porque la serie sintetica ya venia en un rango pequeno y acotado.

In [ ]:
import os
import time
import random
import zipfile
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# semillas fijas para poder reproducir los resultados de una corrida a otra
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# mismo patron que usa el cuadernillo base para elegir GPU si esta disponible
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo de entrenamiento:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM total (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

Dispositivo de entrenamiento: cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
VRAM total (GB): 8.0


## 1. Descarga y carga del dataset

`ElectricityLoadDiagrams20112014` trae el consumo electrico de 370 clientes, medido
cada 15 minutos entre 2011 y 2014, en un unico archivo de texto separado por punto y
coma y con coma como separador decimal (pensado para Excel en configuracion europea).

De la ficha del dataset en UCI conviene recordar:

- No hay valores faltantes.
- Los valores estan en kW por intervalo de 15 minutos.
- Algunos clientes no existian todavia al inicio del periodo y aparecen a cero en esos
  meses.
- Los dos dias de cambio de hora al ano (marzo y octubre) tienen 23 o 25 horas en vez
  de 24, lo cual puede generar algun valor puntual atipico esos dias.

Descargamos el zip oficial y lo dejamos cacheado en disco para no repetir la descarga
de casi 250 MB cada vez que se reinicia el kernel.

In [ ]:
DATA_DIR = "data"
ZIP_PATH = os.path.join(DATA_DIR, "electricityloaddiagrams20112014.zip")
TXT_PATH = os.path.join(DATA_DIR, "LD2011_2014.txt")
DATA_URL = "https://archive.ics.uci.edu/static/public/321/electricityloaddiagrams20112014.zip"

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(TXT_PATH):
    if not os.path.exists(ZIP_PATH):
        print("Descargando dataset desde UCI (puede tardar varios minutos, son ~250 MB)...")
        urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
        print("Descarga terminada.")
    else:
        print("El zip ya estaba descargado, se reutiliza.")

    print("Descomprimiendo...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(DATA_DIR)
    print("Listo.")
else:
    print("El archivo LD2011_2014.txt ya existe en disco, no hace falta descargarlo de nuevo.")

print("Tamano del archivo de datos (MB):", round(os.path.getsize(TXT_PATH) / 1024**2, 1))

El zip ya estaba descargado, se reutiliza.
Descomprimiendo...


BadZipFile: File is not a zip file

In [ ]:
# Alternativa si la descarga directa falla en tu red (por ejemplo detras de un
# proxy corporativo): el paquete oficial ucimlrepo resuelve la descarga solo.
# Se deja comentada porque el metodo de arriba ya es mas liviano en dependencias.
#
# pip install ucimlrepo
# from ucimlrepo import fetch_ucirepo
# electricity = fetch_ucirepo(id=321)
# df_raw = electricity.data.features

In [ ]:
# separador ";" y decimal "," son las particularidades del archivo original.
# la primera columna (sin nombre en el encabezado) es la fecha-hora y la
# usamos como indice, parseandola directamente como fecha
print("Cargando el csv completo, con 370 columnas puede tardar uno o dos minutos...")
t0 = time.time()
df = pd.read_csv(
    TXT_PATH,
    sep=";",
    decimal=",",
    index_col=0,
    parse_dates=True,
)
print(f"Cargado en {time.time() - t0:.1f} segundos.")
df.shape

## 2. Exploracion de datos (EDA)

Antes de definir el problema de prediccion miramos los datos: cuantas filas y columnas
hay, en que rango de fechas se mueven, si hay huecos, y como se ve la serie tanto para
un cliente puntual como para el conjunto agregado.

In [ ]:
print("Filas (marcas de tiempo):", df.shape[0])
print("Columnas (clientes):", df.shape[1])
print("Primera fecha:", df.index.min())
print("Ultima fecha:", df.index.max())
print("Valores nulos totales:", df.isna().sum().sum())

In [ ]:
df.iloc[:, :5].describe()

In [ ]:
# un mes de datos de tres clientes cualquiera, para ver la forma tipica de la
# serie: se nota el patron diario y el patron semanal
clientes_ejemplo = df.columns[:3]
ventana = df.loc["2013-01-01":"2013-01-31", clientes_ejemplo]

plt.figure(figsize=(12, 4))
for col in clientes_ejemplo:
    plt.plot(ventana.index, ventana[col], label=col, linewidth=0.8)
plt.title("Consumo en kW durante enero de 2013, tres clientes de ejemplo")
plt.xlabel("Fecha")
plt.ylabel("kW")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# demanda agregada: suma de los 370 clientes en cada marca de tiempo. Esta va
# a ser la serie que predecimos mas adelante
demanda_total = df.sum(axis=1)
demanda_total.name = "demanda_total_kw"

plt.figure(figsize=(12, 4))
demanda_total.loc["2012-01-01":"2012-03-01"].plot(linewidth=0.8)
plt.title("Demanda total agregada (kW), enero-febrero 2012")
plt.xlabel("Fecha")
plt.ylabel("kW")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

demanda_total.describe()

## 3. Decision de diseño y como se cumplen los requisitos

Trabajar con las 370 series a la vez (prediccion multivariada) agrega complejidad que
no aporta nada a la comparacion que nos interesa (RNN vs LSTM vs GRU). Por eso, igual
que en el cuadernillo base donde se trabajaba con una unica serie sintetica por
muestra, aca reducimos el problema a una sola serie real: la **demanda total
agregada**.

Los datos originales vienen cada 15 minutos, los pasamos a **resolucion horaria**
(promediando los 4 valores de cada hora). Esto reduce el ruido de muy alta frecuencia
y hace que un paso de la ventana temporal equivalga a una hora, algo mas facil de
interpretar.

El problema se plantea igual que el primer ejemplo del cuadernillo base (predecir el
ultimo valor de la serie a partir del historico): a partir de las ultimas `n` horas de
demanda, predecimos la demanda de la hora siguiente.

### Como se cumplen los requisitos del enunciado

- `n` (tamano de la ventana de entrada, equivalente al `n_steps` del cuadernillo base):
  usamos **`n = 24`** horas de historial (un dia completo), que cumple `n > 20`.
- Cada ventana de entrada `X[i]` tiene asociado exactamente un valor objetivo `y[i]`
  (el siguiente), asi que por construccion **`m = o`**: el numero de filas de `X` es
  igual al numero de filas de `y`.
- Con aproximadamente 4 anos de datos horarios (unas 35.000 horas) y una ventana de 24,
  el numero de muestras que se generan queda **muy por encima de 20.000**. No lo damos
  por hecho: lo verificamos con codigo justo despues de construir las ventanas.

In [ ]:
# remuestreo horario: promediamos los 4 valores de cada hora
serie_horaria = demanda_total.resample("1h").mean()

print("Muestras horarias totales:", len(serie_horaria))
print("Desde", serie_horaria.index.min(), "hasta", serie_horaria.index.max())
print("Valores nulos tras el remuestreo:", serie_horaria.isna().sum())

plt.figure(figsize=(12, 4))
serie_horaria.plot(linewidth=0.5)
plt.title("Demanda total agregada, resolucion horaria, serie completa")
plt.xlabel("Fecha")
plt.ylabel("kW")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
N_STEPS = 24  # tamano de la ventana de entrada (equivale al n_steps del cuadernillo base), n > 20

def construir_ventanas(serie, n_steps):
    '''
    Version del cuadernillo base adaptada a datos reales: en vez de generar
    series sinteticas con generate_time_series y despues recortar
    series[:, :n_steps] / series[:, -1], aca recorremos la unica serie real
    con una ventana deslizante para producir el mismo tipo de estructura:
      X: bloques de n_steps valores consecutivos, forma (m, n_steps)
      y: el valor que sigue a cada bloque, forma (m,)
    '''
    valores = serie.values.astype(np.float32)
    X = np.lib.stride_tricks.sliding_window_view(valores[:-1], n_steps)
    y = valores[n_steps:]
    return X, y

X, y = construir_ventanas(serie_horaria, N_STEPS)

m, n = X.shape
o = y.shape[0]
print(f"n (tamano de la ventana) = {n}")
print(f"m (muestras de entrada, filas de X) = {m}")
print(f"o (muestras de salida, filas de y) = {o}")

assert n > 20, "n deberia ser mayor a 20"
assert m == o, "m y o deberian coincidir, cada ventana tiene un unico valor objetivo"
assert m > 20000, "se necesitan mas de 20.000 muestras"
print("Requisitos de n, m y o verificados correctamente.")

## 4. Particion temporal (train / eval / test) y normalizacion

Igual que en el cuadernillo base, dividimos en tres particiones: `train`, `eval` (la
que ahi se llamaba validacion) y `test`. La diferencia importante frente al ejemplo
sintetico es que aca **no podemos partir al azar**: en series temporales hay que
respetar el orden cronologico, cortando la serie en tres tramos consecutivos, porque
mezclar el orden dejaria que el modelo viera indirectamente informacion del futuro
durante el entrenamiento.

Usamos una particion 70 % / 15 % / 15 %.

Para la normalizacion: el `scaler` se ajusta solo con los datos de `train`, y ese mismo
`scaler` (sin volver a ajustarlo) se aplica despues a `eval` y a `test`. Si lo
ajustaramos con todo el dataset estariamos filtrando informacion del futuro hacia el
entrenamiento.

In [ ]:
n_train = int(m * 0.70)
n_eval = int(m * 0.15)
n_test = m - n_train - n_eval

X_train, y_train = X[:n_train], y[:n_train]
X_eval, y_eval = X[n_train:n_train + n_eval], y[n_train:n_train + n_eval]
X_test, y_test = X[n_train + n_eval:], y[n_train + n_eval:]

print("Train:", X_train.shape, y_train.shape)
print("Eval: ", X_eval.shape, y_eval.shape)
print("Test: ", X_test.shape, y_test.shape)

In [ ]:
# StandardScaler ajustado solo con train. X e y comparten la misma escala
# porque son la misma variable (demanda) en distintos instantes de tiempo
scaler = StandardScaler()
scaler.fit(X_train.reshape(-1, 1))

def escalar(array_2d):
    forma_original = array_2d.shape
    return scaler.transform(array_2d.reshape(-1, 1)).reshape(forma_original).astype(np.float32)

X_train_s, X_eval_s, X_test_s = escalar(X_train), escalar(X_eval), escalar(X_test)
y_train_s = scaler.transform(y_train.reshape(-1, 1)).astype(np.float32).reshape(-1)
y_eval_s = scaler.transform(y_eval.reshape(-1, 1)).astype(np.float32).reshape(-1)
y_test_s = scaler.transform(y_test.reshape(-1, 1)).astype(np.float32).reshape(-1)

print("Media usada por el scaler:", scaler.mean_[0])
print("Desviacion estandar usada por el scaler:", scaler.scale_[0])

## 5. `TimeSeriesDataset`, `dataset` y `dataloader`

Esta es exactamente la misma clase `TimeSeriesDataset` del cuadernillo
`02_rnn_forecast`: recibe `X` y opcionalmente `y`, y en `__getitem__` devuelve el par
`(X[ix], y[ix])` convertido a tensores si `train=True`, o solo `X[ix]` si `train=False`
(pensado para el caso en que se quiere predecir sin tener las etiquetas a mano).

Antes de crear el `Dataset` le damos a `X` una dimension extra al final, para que
quede con forma `(m, n_steps, 1)`: esa ultima dimension es el numero de variables por
paso de tiempo (aca solo tenemos una, la demanda), y es la forma que espera
`torch.nn.RNN` / `LSTM` / `GRU` con `batch_first=True`. Es el mismo `[..., np.newaxis]`
que usaba `generate_time_series` en el cuadernillo original.

In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y=None, train=True):
        # agregamos la dimension de "numero de variables por paso de tiempo"
        # (1 en nuestro caso), igual que hacia generate_time_series con
        # [..., np.newaxis] en el cuadernillo original
        self.X = X[..., np.newaxis] if X.ndim == 2 else X
        self.y = y[..., np.newaxis] if (y is not None and y.ndim == 1) else y
        self.train = train

    def __len__(self):
        return len(self.X)

    def __getitem__(self, ix):
        if self.train:
            return torch.from_numpy(self.X[ix]), torch.from_numpy(self.y[ix])
        return torch.from_numpy(self.X[ix])


dataset = {
    "train": TimeSeriesDataset(X_train_s, y_train_s),
    "eval": TimeSeriesDataset(X_eval_s, y_eval_s),
    "test": TimeSeriesDataset(X_test_s, y_test_s, train=False),
}

BATCH_SIZE = 128

dataloader = {
    "train": DataLoader(dataset["train"], shuffle=True, batch_size=BATCH_SIZE),
    "eval": DataLoader(dataset["eval"], shuffle=False, batch_size=BATCH_SIZE),
    "test": DataLoader(dataset["test"], shuffle=False, batch_size=BATCH_SIZE),
}

x_batch, y_batch = dataset["train"][0:4]
print("Forma de X de ejemplo:", x_batch.shape, "-> (batch, n_steps, num_variables)")
print("Forma de y de ejemplo:", y_batch.shape)

## 6. `plot_series`, adaptada a datos reales

En el cuadernillo original, `plot_series` dibuja una grilla de 3x5 ejemplos, con el
historico en azul, el valor real a predecir marcado con una `x`, y (si se le pasa) la
prediccion del modelo marcada con un circulo rojo. Tambien soporta pintar una banda de
desviacion estandar alrededor de la prediccion, que se usa mas adelante para los
intervalos de confianza.

La unica diferencia real frente a la version original es que ahi los ejes estaban
fijados a un rango `[-1, 1]` porque la serie sintetica siempre se movia en ese rango.
Con datos reales de demanda electrica eso no aplica, asi que calculamos los limites de
los ejes a partir de los datos que se esten graficando en cada llamada.

In [ ]:
def plot_series(series, y=None, y_pred=None, y_pred_std=None, x_label="paso de tiempo", y_label="demanda"):
    r, c = 3, 5
    fig, axes = plt.subplots(nrows=r, ncols=c, sharey=True, sharex=True, figsize=(20, 10))

    # rango de valores para los ejes, calculado a partir de lo que se va a
    # graficar (en el original estaba fijo a [-1, 1] porque la serie sintetica
    # siempre se movia en ese rango)
    piezas = [series]
    if y is not None:
        piezas.append(y)
    if y_pred is not None:
        piezas.append(y_pred)
    todo = np.concatenate([p.reshape(-1) for p in piezas])
    y_min, y_max = todo.min(), todo.max()
    margen = (y_max - y_min) * 0.1 + 1e-6

    for row in range(r):
        for col in range(c):
            plt.sca(axes[row][col])
            ix = col + row * c
            if ix >= len(series):
                plt.axis("off")
                continue
            plt.plot(series[ix, :], ".-", markersize=3, linewidth=0.8)
            if y is not None:
                plt.plot(range(len(series[ix, :]), len(series[ix, :]) + len(y[ix])), y[ix], "bx", markersize=10)
            if y_pred is not None:
                plt.plot(range(len(series[ix, :]), len(series[ix, :]) + len(y_pred[ix])), y_pred[ix], "ro")
            if y_pred_std is not None:
                plt.plot(range(len(series[ix, :]), len(series[ix, :]) + len(y_pred[ix])), y_pred[ix] + y_pred_std[ix], "r--", linewidth=0.6)
                plt.plot(range(len(series[ix, :]), len(series[ix, :]) + len(y_pred[ix])), y_pred[ix] - y_pred_std[ix], "r--", linewidth=0.6)
            plt.grid(True, alpha=0.3)
            plt.axis([0, len(series[ix, :]) + (len(y[ix]) if y is not None else 0), y_min - margen, y_max + margen])
            if x_label and row == r - 1:
                plt.xlabel(x_label, fontsize=11)
            if y_label and col == 0:
                plt.ylabel(y_label, fontsize=11)

    plt.tight_layout()
    plt.show()

# probamos la funcion con quince ejemplos del set de test, sin predicciones todavia
plot_series(X_test_s[:15], y_test_s[:15].reshape(-1, 1))

## 7. Modelo generico: RNN, LSTM o GRU

El cuadernillo base define primero una clase `RNN` sencilla (una capa recurrente mas
una capa lineal) y despues una `DeepRNN` con `num_layers=2`. Ac creamos una unica
clase generica, `RedRecurrente`, que recibe como parametro que tipo de celda usar
(`"rnn"`, `"lstm"` o `"gru"`), para poder instanciar las tres arquitecturas con
exactamente el mismo codigo y compararlas en igualdad de condiciones. La logica del
`forward` es identica a la del cuadernillo original: `x, h = self.rnn(x)` seguido de
`self.fc(x[:, -1])`, quedandonos solo con la salida del ultimo paso temporal.

Si se usa mas de una capa (`num_layers > 1`), se puede pasar `dropout` para regularizar
entre capas recurrentes, tal como se hace en la seccion de intervalos de confianza del
cuadernillo original.

In [ ]:
class RedRecurrente(torch.nn.Module):
    def __init__(self, celda="lstm", hidden_size=32, num_layers=1, dropout=0.0, n_out=1):
        super().__init__()
        celdas_disponibles = {"rnn": torch.nn.RNN, "lstm": torch.nn.LSTM, "gru": torch.nn.GRU}
        assert celda in celdas_disponibles, f"celda debe ser una de {list(celdas_disponibles)}"

        kwargs_celda = dict(input_size=1, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        # el argumento dropout de nn.RNN/LSTM/GRU solo tiene efecto real con
        # mas de una capa; con una sola capa pytorch avisa con un warning si
        # se pasa igualmente, asi que lo agregamos solo cuando corresponde
        if num_layers > 1:
            kwargs_celda["dropout"] = dropout

        self.rnn = celdas_disponibles[celda](**kwargs_celda)
        self.fc = torch.nn.Linear(hidden_size, n_out)
        self.celda = celda

    def forward(self, x):
        # x: (batch, n_steps, 1)
        # salida de la celda recurrente en cada paso de tiempo, y el estado
        # oculto final (que en LSTM es una tupla (h_n, c_n) y en RNN/GRU es
        # un unico tensor; aca no lo necesitamos, solo nos interesa la salida)
        x, h = self.rnn(x)
        y = self.fc(x[:, -1])  # nos quedamos con el ultimo paso temporal
        return y


def construir_modelo(celda, hidden_size=32, num_layers=1, dropout=0.0, n_out=1):
    return RedRecurrente(celda=celda, hidden_size=hidden_size, num_layers=num_layers, dropout=dropout, n_out=n_out).to(device)

## 8. Validando la red antes de entrenar en serio

Antes de lanzar un entrenamiento largo seguimos los pasos que pide la guia (seccion 2,
"Validar la red antes de entrenar en serio"): comprobar que las dimensiones de entrada
y salida son las esperadas, y comprobar que el modelo puede memorizar primero una sola
muestra y despues un unico batch. Si algo de esto fallara seria senal de un error de
implementacion que conviene detectar ahora, no despues de un entrenamiento largo.

In [ ]:
# 8.1 comprobacion de dimensiones
modelo_prueba = construir_modelo("lstm", hidden_size=16)
entrada_prueba = torch.randn(64, N_STEPS, 1).to(device)
salida_prueba = modelo_prueba(entrada_prueba)
print("Forma de entrada: ", entrada_prueba.shape)
print("Forma de salida:  ", salida_prueba.shape, "-> se espera (64, 1)")
assert salida_prueba.shape == (64, 1)
print("Dimensiones correctas.")

In [ ]:
# 8.2 fit de una sola muestra: si el modelo no logra bajar la perdida a
# practicamente cero memorizando un unico ejemplo, algo esta mal
x_una, y_una = dataset["train"][0]
x_una = x_una.unsqueeze(0).to(device)
y_una = y_una.unsqueeze(0).to(device)

modelo_prueba = construir_modelo("lstm", hidden_size=16)
criterio = torch.nn.MSELoss()
optimizador_prueba = torch.optim.Adam(modelo_prueba.parameters(), lr=0.01)

for epoca in range(1, 201):
    pred = modelo_prueba(x_una)
    perdida = criterio(pred, y_una)
    optimizador_prueba.zero_grad()
    perdida.backward()
    optimizador_prueba.step()
    if epoca % 50 == 0:
        print(f"epoca {epoca}, perdida {perdida.item():.6f}")

print("Si la perdida final es cercana a 0, el modelo puede memorizar una muestra sin problema.")

In [ ]:
# 8.3 fit de un solo batch, con un lr algo mas agresivo para ver el
# sobreajuste rapido en pocas epocas (misma logica de la guia)
loader_un_batch = DataLoader(dataset["train"], batch_size=BATCH_SIZE, shuffle=True)
x_batch_prueba, y_batch_prueba = next(iter(loader_un_batch))
x_batch_prueba, y_batch_prueba = x_batch_prueba.to(device), y_batch_prueba.to(device)

modelo_prueba = construir_modelo("lstm", hidden_size=16)
optimizador_prueba = torch.optim.Adam(modelo_prueba.parameters(), lr=0.01)

for epoca in range(1, 301):
    pred = modelo_prueba(x_batch_prueba)
    perdida = criterio(pred, y_batch_prueba)
    optimizador_prueba.zero_grad()
    perdida.backward()
    optimizador_prueba.step()
    if epoca % 100 == 0:
        print(f"epoca {epoca}, perdida {perdida.item():.6f}")

print("La perdida deberia bajar de forma clara y sostenida, senal de que el bucle de entrenamiento esta bien planteado.")

## 9. Funcion `fit`, ampliada respecto al cuadernillo base

La funcion `fit` del cuadernillo original es deliberadamente simple: entrena por
epocas, calcula la perdida de entrenamiento y de `eval`, y muestra todo en una barra de
progreso con `tqdm`. No guarda checkpoints, no tiene early stopping ni scheduler, y no
aplica ninguna regularizacion mas alla de la que traiga la arquitectura.

Mantenemos esa misma base (la barra `tqdm`, la estructura del bucle, los nombres
`train_loss` / `eval_loss`) y le sumamos lo que pide la guia de buenas practicas:

- **Checkpointing:** se guarda el modelo cada vez que mejora la perdida de `eval`, y al
  terminar se recargan esos pesos (no los ultimos, que podrian ya estar en
  sobreajuste).
- **Early stopping:** opcional, corta el entrenamiento si `eval` no mejora durante un
  numero de epocas seguidas.
- **Scheduler:** opcional, para variar el learning rate durante el entrenamiento.
- **Weight decay (regularizacion L2):** se aplica en el optimizador que se le pasa a
  esta funcion desde afuera, siguiendo el mismo criterio que la guia.
- **Clipping de gradiente:** no estaba en el cuadernillo original, pero es una practica
  casi obligatoria al entrenar redes recurrentes (la RNN simple en particular es
  propensa al problema de gradientes que explotan al propagarse hacia atras en el
  tiempo). Limitar la norma del gradiente evita que un paso de optimizacion "se
  dispare" y arruine los pesos ya aprendidos.

In [ ]:
def fit(model, dataloader, optimizer, scheduler=None, epochs=10, early_stopping=0,
        clip_grad_norm=1.0, ckpt_path="ckpt.pt", target_ultimo_paso=False):
    model.to(device)
    criterion = torch.nn.MSELoss()

    historia = {"epoch": [], "loss": [], "eval_loss": [], "eval_mae": [], "lr": []}
    mejor_eval_loss = float("inf")
    paciencia = 0

    bar = tqdm(range(1, epochs + 1))
    for epoch in bar:
        # -------- entrenamiento (mismo esqueleto que el cuadernillo original) --------
        model.train()
        train_loss = []
        for batch in dataloader["train"]:
            X, y = batch
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            y_hat = model(X)
            loss = criterion(y_hat, y)
            loss.backward()
            if clip_grad_norm:
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad_norm)
            optimizer.step()
            train_loss.append(loss.item())

        # -------- evaluacion --------
        model.eval()
        eval_loss, predicciones, reales = [], [], []
        with torch.no_grad():
            for batch in dataloader["eval"]:
                X, y = batch
                X, y = X.to(device), y.to(device)
                y_hat = model(X)
                eval_loss.append(criterion(y_hat, y).item())
                predicciones.append(y_hat.cpu().numpy())
                reales.append(y.cpu().numpy())

        loss_media = float(np.mean(train_loss))
        eval_loss_media = float(np.mean(eval_loss))
        eval_mae = float(mean_absolute_error(np.concatenate(reales).reshape(-1), np.concatenate(predicciones).reshape(-1)))

        historia["epoch"].append(epoch)
        historia["loss"].append(loss_media)
        historia["eval_loss"].append(eval_loss_media)
        historia["eval_mae"].append(eval_mae)
        historia["lr"].append(optimizer.param_groups[0]["lr"])

        # checkpoint del mejor modelo visto hasta ahora, segun eval_loss
        if eval_loss_media < mejor_eval_loss:
            mejor_eval_loss = eval_loss_media
            torch.save(model.state_dict(), ckpt_path)
            paciencia = 0
        else:
            paciencia += 1

        if scheduler is not None:
            scheduler.step()

        bar.set_description(f"loss {loss_media:.5f} eval_loss {eval_loss_media:.5f} eval_mae {eval_mae:.5f}")

        if early_stopping and paciencia > early_stopping:
            print(f"Entrenamiento detenido en la epoca {epoch}: {early_stopping} epocas seguidas sin mejorar en eval.")
            break

    # recargamos los pesos del mejor checkpoint, no los ultimos calculados
    model.load_state_dict(torch.load(ckpt_path))
    return historia

## 10. Funcion `predict`

Identica en espiritu a la del cuadernillo base: recorre un `dataloader` sin etiquetas
(`train=False` en el `TimeSeriesDataset`) y va concatenando las predicciones del
modelo en un unico tensor.

In [ ]:
def predict(model, dataloader):
    model.eval()
    with torch.no_grad():
        preds = torch.tensor([]).to(device)
        for batch in dataloader:
            X = batch
            X = X.to(device)
            pred = model(X)
            preds = torch.cat([preds, pred])
        return preds

## 11. Iterando rapido: subconjunto representativo y busqueda de hiperparametros

Igual que propone la guia (y a diferencia del cuadernillo original, que entrenaba
directamente sobre todo el dataset porque era pequeno y sintetico), aca conviene
primero probar combinaciones de learning rate y batch size sobre un subconjunto chico
del set de entrenamiento, y recien despues repetir el mejor entrenamiento con todos
los datos. Hacemos la busqueda con una **LSTM** como arquitectura de referencia, y
despues usamos los mismos hiperparametros para las tres arquitecturas, para que la
comparacion final sea justa.

In [ ]:
N_SUBSET = 4000
subset_train = TimeSeriesDataset(X_train_s[:N_SUBSET], y_train_s[:N_SUBSET])

espacio_lr = [0.01, 0.005, 0.001, 0.0005, 0.0001]
espacio_bs = [32, 64, 128]

N_PRUEBAS = 6
resultados_busqueda = []

for i in range(N_PRUEBAS):
    lr = random.choice(espacio_lr)
    bs = random.choice(espacio_bs)
    print(f"Prueba {i + 1}/{N_PRUEBAS}: lr={lr}, batch_size={bs}")

    dl_prueba = {
        "train": DataLoader(subset_train, batch_size=bs, shuffle=True),
        "eval": dataloader["eval"],
    }

    modelo = construir_modelo("lstm", hidden_size=64, num_layers=1)
    optimizador = torch.optim.AdamW(modelo.parameters(), lr=lr, weight_decay=1e-4)

    historia = fit(modelo, dl_prueba, optimizador, epochs=15, ckpt_path=f"ckpt_busqueda_{i}.pt")
    resultados_busqueda.append({"lr": lr, "bs": bs, "historia": historia})

In [ ]:
plt.figure(figsize=(9, 4))
for r in resultados_busqueda:
    etiqueta = f"lr={r['lr']}, bs={r['bs']}"
    plt.plot(r["historia"]["epoch"], r["historia"]["eval_loss"], label=etiqueta)
plt.xlabel("epoca")
plt.ylabel("eval_loss (MSE, escala normalizada)")
plt.title("Random search de hiperparametros sobre el subconjunto (arquitectura LSTM)")
plt.legend(fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

mejor_combinacion = min(resultados_busqueda, key=lambda r: r["historia"]["eval_loss"][-1])
LR_ELEGIDO = mejor_combinacion["lr"]
BS_ELEGIDO = mejor_combinacion["bs"]
print(f"Mejor combinacion encontrada: lr={LR_ELEGIDO}, batch_size={BS_ELEGIDO}")

## 12. Entrenamiento final: RNN, LSTM y GRU sobre el dataset completo

Con los hiperparametros ya elegidos, entrenamos las tres arquitecturas sobre el
dataset completo, cada una con exactamente la misma configuracion (mismo optimizador,
mismo weight decay, mismo dropout, mismo early stopping), para que la unica variable
que cambie sea el tipo de celda recurrente.

Resumen de lo que se aplica a las tres:

- **Optimizador:** `AdamW` con `lr` y `batch_size` elegidos en la busqueda anterior.
- **Scheduler:** `StepLR`, reduce el learning rate a la mitad cada 10 epocas.
- **Regularizacion:** `weight_decay` en el optimizador, `dropout` entre capas
  recurrentes (usamos 2 capas para que el dropout tenga efecto) y **early stopping**
  con paciencia de 15 epocas.
- **Estabilidad del entrenamiento:** clipping de gradiente, especialmente relevante
  para la RNN simple.
- **Presupuesto de computo:** con una unica variable de entrada y ventanas de 24
  pasos, `hidden_size=64` es mas que suficiente y las tres arquitecturas entrenan
  comodas dentro de los 8 GB de VRAM de una RTX 4070.

In [ ]:
EPOCHS_FINAL = 60
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.2
WEIGHT_DECAY = 1e-4
EARLY_STOPPING = 15

dataloader_final = {
    "train": DataLoader(dataset["train"], batch_size=BS_ELEGIDO, shuffle=True),
    "eval": dataloader["eval"],
}

arquitecturas = ["rnn", "lstm", "gru"]
modelos_entrenados = {}
historias = {}

for arq in arquitecturas:
    print("=" * 70)
    print(f"Entrenando arquitectura: {arq.upper()}")
    print("=" * 70)

    modelo = construir_modelo(arq, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, dropout=DROPOUT)
    optimizador = torch.optim.AdamW(modelo.parameters(), lr=LR_ELEGIDO, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizador, step_size=10, gamma=0.5)

    t0 = time.time()
    historia = fit(
        modelo, dataloader_final, optimizador, scheduler=scheduler,
        epochs=EPOCHS_FINAL, early_stopping=EARLY_STOPPING,
        clip_grad_norm=1.0, ckpt_path=f"ckpt_{arq}.pt",
    )
    duracion = time.time() - t0
    print(f"Tiempo de entrenamiento ({arq}): {duracion:.1f} segundos")

    modelos_entrenados[arq] = modelo
    historias[arq] = historia

## 13. Comparacion de resultados

Comparamos las tres arquitecturas en tres niveles: las curvas de perdida durante el
entrenamiento, las metricas finales sobre el conjunto de **test** (que ningun modelo
vio ni durante el entrenamiento ni durante la busqueda de hiperparametros) ya
convertidas de vuelta a kW, y una inspeccion visual con la misma `plot_series` del
cuadernillo original.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
for arq in arquitecturas:
    plt.plot(historias[arq]["epoch"], historias[arq]["loss"], label=f"{arq} (train)")
plt.xlabel("epoca")
plt.ylabel("MSE (escala normalizada)")
plt.title("Perdida de entrenamiento")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
for arq in arquitecturas:
    plt.plot(historias[arq]["epoch"], historias[arq]["eval_loss"], label=f"{arq} (eval)")
plt.xlabel("epoca")
plt.ylabel("MSE (escala normalizada)")
plt.title("Perdida de evaluacion")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
def evaluar_en_test(modelo, dataloader_test, y_test_original, scaler):
    y_pred = predict(modelo, dataloader_test).cpu().numpy().reshape(-1, 1)
    y_pred_kw = scaler.inverse_transform(y_pred).reshape(-1)
    y_real_kw = y_test_original

    mae = mean_absolute_error(y_real_kw, y_pred_kw)
    rmse = np.sqrt(mean_squared_error(y_real_kw, y_pred_kw))
    mape = float(np.mean(np.abs((y_real_kw - y_pred_kw) / y_real_kw))) * 100

    return {"mae": mae, "rmse": rmse, "mape": mape, "pred": y_pred_kw, "real": y_real_kw}


resultados_test = {}
for arq in arquitecturas:
    resultados_test[arq] = evaluar_en_test(modelos_entrenados[arq], dataloader["test"], y_test, scaler)

tabla_comparativa = pd.DataFrame({
    arq: {
        "MAE (kW)": resultados_test[arq]["mae"],
        "RMSE (kW)": resultados_test[arq]["rmse"],
        "MAPE (%)": resultados_test[arq]["mape"],
    }
    for arq in arquitecturas
}).T

tabla_comparativa

In [ ]:
# misma logica visual que plot_series del cuadernillo original: quince
# ejemplos de test, historico en azul, valor real con "x", prediccion de cada
# arquitectura superpuesta
for arq in arquitecturas:
    print(f"Arquitectura: {arq.upper()}")
    y_pred_arq = resultados_test[arq]["pred"][:15].reshape(-1, 1)
    plot_series(X_test[:15], y_test[:15].reshape(-1, 1), y_pred_arq)

In [ ]:
# comparacion en una unica grafica: una semana del conjunto de test, real
# contra la prediccion de cada arquitectura
TRAMO = slice(0, 24 * 7)

plt.figure(figsize=(13, 5))
plt.plot(resultados_test["rnn"]["real"][TRAMO], label="Real", color="black", linewidth=1.5)
for arq in arquitecturas:
    plt.plot(resultados_test[arq]["pred"][TRAMO], label=f"Prediccion {arq.upper()}", linewidth=1.0, alpha=0.85)
plt.xlabel("Horas dentro del tramo de test mostrado")
plt.ylabel("Demanda total (kW)")
plt.title("Prediccion a un paso, real vs modelos, una semana del conjunto de test")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 14. Extension: prediccion a varias horas hacia adelante

El cuadernillo base no se queda solo con predecir un unico valor: tambien muestra como
extender el modelo para predecir varios pasos hacia adelante (ahi, los siguientes 10
valores de la serie sintetica), cambiando la capa final para que en vez de devolver un
unico numero devuelva `n_out` numeros.

Aplicamos exactamente la misma idea a nuestros datos reales: en vez de predecir solo
la hora siguiente, predecimos las siguientes **`H = 6` horas** de demanda a partir de
las 24 horas anteriores. Reconstruimos `Y` de la misma forma que en el cuadernillo
original (para cada ventana, apilamos los `H` valores que le siguen), y reusamos la
misma clase `RedRecurrente` cambiando `n_out=H`.

In [ ]:
H = 6  # horas a predecir hacia adelante

def construir_ventanas_multistep(serie, n_steps, h):
    valores = serie.values.astype(np.float32)
    total = len(valores) - n_steps - h + 1
    X = np.lib.stride_tricks.sliding_window_view(valores[: n_steps + total - 1], n_steps)[:total]
    Y = np.stack([valores[n_steps + i: n_steps + i + h] for i in range(total)])
    return X, Y

X_ms, Y_ms = construir_ventanas_multistep(serie_horaria, N_STEPS, H)
print("X multistep:", X_ms.shape, " Y multistep:", Y_ms.shape)

# mismos cortes 70/15/15, respetando el orden temporal
m_ms = len(X_ms)
n_train_ms = int(m_ms * 0.70)
n_eval_ms = int(m_ms * 0.15)

X_train_ms, Y_train_ms = X_ms[:n_train_ms], Y_ms[:n_train_ms]
X_eval_ms, Y_eval_ms = X_ms[n_train_ms:n_train_ms + n_eval_ms], Y_ms[n_train_ms:n_train_ms + n_eval_ms]
X_test_ms, Y_test_ms = X_ms[n_train_ms + n_eval_ms:], Y_ms[n_train_ms + n_eval_ms:]

# reutilizamos el mismo scaler ya ajustado sobre train (misma variable, misma escala)
X_train_ms_s = escalar(X_train_ms)
X_eval_ms_s = escalar(X_eval_ms)
X_test_ms_s = escalar(X_test_ms)
Y_train_ms_s = escalar(Y_train_ms)
Y_eval_ms_s = escalar(Y_eval_ms)
Y_test_ms_s = escalar(Y_test_ms)

dataset_ms = {
    "train": TimeSeriesDataset(X_train_ms_s, Y_train_ms_s),
    "eval": TimeSeriesDataset(X_eval_ms_s, Y_eval_ms_s),
    "test": TimeSeriesDataset(X_test_ms_s, Y_test_ms_s, train=False),
}

dataloader_ms = {
    "train": DataLoader(dataset_ms["train"], batch_size=BS_ELEGIDO, shuffle=True),
    "eval": DataLoader(dataset_ms["eval"], batch_size=BATCH_SIZE, shuffle=False),
    "test": DataLoader(dataset_ms["test"], batch_size=BATCH_SIZE, shuffle=False),
}

In [ ]:
# entrenamos una GRU multistep como ejemplo (misma arquitectura generica,
# ahora con n_out=H en vez de n_out=1); menos epocas porque el objetivo es
# ilustrar la tecnica, no repetir la busqueda de hiperparametros completa
modelo_multistep = construir_modelo("gru", hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, dropout=DROPOUT, n_out=H)
optimizador_ms = torch.optim.AdamW(modelo_multistep.parameters(), lr=LR_ELEGIDO, weight_decay=WEIGHT_DECAY)
scheduler_ms = torch.optim.lr_scheduler.StepLR(optimizador_ms, step_size=10, gamma=0.5)

historia_ms = fit(
    modelo_multistep, dataloader_ms, optimizador_ms, scheduler=scheduler_ms,
    epochs=30, early_stopping=10, clip_grad_norm=1.0, ckpt_path="ckpt_gru_multistep.pt",
)

In [ ]:
y_pred_ms = predict(modelo_multistep, dataloader_ms["test"]).cpu().numpy()
y_pred_ms_kw = scaler.inverse_transform(y_pred_ms.reshape(-1, 1)).reshape(y_pred_ms.shape)

mae_ms = mean_absolute_error(Y_test_ms.reshape(-1), y_pred_ms_kw.reshape(-1))
print(f"MAE prediciendo las proximas {H} horas de una sola vez: {mae_ms:.2f} kW")

plot_series(X_test_ms[:15], Y_test_ms[:15], y_pred_ms_kw[:15])

## 15. Extension: intervalos de confianza con dropout en la inferencia

El cuadernillo original cierra con una tecnica simple para obtener intervalos de
confianza: dejar el `dropout` **activo** durante la inferencia (llamando
`model.train()` en vez de `model.eval()`), repetir la prediccion muchas veces con una
configuracion distinta de neuronas apagadas cada vez, y usar la media de todas esas
predicciones como resultado final y la desviacion estandar como medida de
incertidumbre. Es la tecnica conocida como *MC-Dropout* (Monte Carlo Dropout).

La aplicamos sobre el modelo GRU multistep que acabamos de entrenar, que ya tiene
`dropout=0.2` entre sus capas recurrentes.

In [ ]:
def predict_con_incertidumbre(model, dataloader, n_muestras=50):
    # activamos dropout durante la inferencia a proposito: normalmente aca
    # iria model.eval(), pero para MC-Dropout necesitamos que el dropout
    # siga funcionando en cada llamada
    model.train()
    predicciones = []
    with torch.no_grad():
        for _ in range(n_muestras):
            preds = torch.tensor([]).to(device)
            for batch in dataloader:
                X = batch.to(device)
                pred = model(X)
                preds = torch.cat([preds, pred])
            predicciones.append(preds.cpu().numpy())
    predicciones = np.stack(predicciones)
    media = predicciones.mean(axis=0)
    desviacion = predicciones.std(axis=0)
    return media, desviacion


media_pred, std_pred = predict_con_incertidumbre(modelo_multistep, dataloader_ms["test"], n_muestras=50)
media_pred_kw = scaler.inverse_transform(media_pred.reshape(-1, 1)).reshape(media_pred.shape)
# la desviacion estandar se escala por el mismo factor que usa el scaler
# para volver a la escala original, sin desplazarla (no le corresponde restar la media)
std_pred_kw = std_pred * scaler.scale_[0]

plot_series(X_test_ms[:15], Y_test_ms[:15], media_pred_kw[:15], std_pred_kw[:15])

## 16. Conclusiones

Algunas lecturas de esta comparacion (los numeros exactos van a variar segun la
semilla, el hardware y cuantas epocas alcance a correr cada modelo antes de que actue
el early stopping, pero el patron general suele repetirse):

- La **RNN simple** suele ser la que mas le cuesta, tanto en velocidad de convergencia
  como en el error final. Es la arquitectura mas sensible al problema de gradientes
  que se desvanecen o explotan al propagarse hacia atras en el tiempo. El clipping de
  gradiente ayuda a que el entrenamiento no se dispare, pero no resuelve el problema de
  fondo: le cuesta mas retener informacion de muchos pasos atras.
- **LSTM** y **GRU** suelen quedar bastante parejas en el error final para un problema
  como este (una unica serie, ventana de 24 pasos), porque las dos resuelven el mismo
  problema de fondo con mecanismos de compuertas parecidos. La diferencia mas notoria
  suele estar en el tiempo de entrenamiento: GRU tiene menos parametros que LSTM (no
  tiene una celda de memoria separada del estado oculto), por lo que cada epoca suele
  ser un poco mas rapida sin perder demasiada capacidad de prediccion.
- El **early stopping** termina decidiendo, en la practica, cuantas epocas entrena
  cada arquitectura: si una converge mas rapido que las otras, el entrenamiento se
  corta antes para esa arquitectura en particular. Eso tambien forma parte de la
  comparacion, no solo importa el error final sino cuanto costo llegar ahi.
- Extender el modelo a prediccion multi-paso (seccion 14) empeora el error por hora
  predicha respecto al modelo de un solo paso, algo esperable: predecir 6 horas de
  una sola vez es un problema mas dificil que predecir solo la siguiente. Esto es
  justamente lo que se discute en el cuadernillo original al comparar predecir un
  unico valor contra encadenar predicciones.
- Los intervalos de confianza obtenidos con MC-Dropout (seccion 15) dan una nocion util
  de cuanto confiar en cada prediccion puntual, aunque hay que tener presente que solo
  capturan la incertidumbre asociada al propio modelo (que tan seguro esta de sus
  pesos), no la incertidumbre inherente a la serie real, que puede ser mayor.

En cuanto al presupuesto de computo: con esta configuracion (una sola serie, ventanas
de 24 pasos, `hidden_size=64`) las tres arquitecturas entrenan rapido incluso en una
GPU de gama media como la RTX 4070 de 8 GB, y corren razonablemente incluso en CPU si
hiciera falta. El cuello de botella real en un problema de este tamano casi siempre es
la carga inicial del csv de 370 columnas, no el entrenamiento en si.